<a href="https://colab.research.google.com/github/WamsyJ/Scaler/blob/main/Model_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Suppress scientific notation
np.set_printoptions(suppress=True)

# Generate randomly distributed parameters
params = np.random.uniform(low=-50, high=150, size=20)

# Make sure important values are at the beginning for better debugging
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

# Round each number to the second decimal place
params = np.round(params, 2)

# Print the parameters
print(params)

[142.8  -49.78   0.   105.22  16.48 -18.28 -19.5  -48.78  32.92  77.53
 141.8   42.16 119.59   0.16 115.18  -8.   -23.92 107.23  23.84  40.68]


In [ ]:
def clamp(params_q: np.array, lower_bound: int, upper_bound: int) -> np.array:
    params_q[params_q < lower_bound] = lower_bound
    params_q[params_q > upper_bound] = upper_bound
    return params_q

def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array, float, int]:
    # Calculate the scale and zero point
    alpha = np.max(params)
    beta = np.min(params)
    scale = (alpha - beta) / (2**bits-1)
    zero = -1*np.round(beta / scale)
    lower_bound, upper_bound = 0, 2**bits-1
    # Quantize the parameters
    quantized = clamp(np.round(params / scale + zero), lower_bound, upper_bound).astype(np.int32)
    return quantized, scale, zero

def asymmetric_dequantize(params_q: np.array, scale: float, zero: int) -> np.array:
    return (params_q - zero) * scale

def symmetric_dequantize(params_q: np.array, scale: float) -> np.array:
    return params_q * scale

def symmetric_quantization(params: np.array, bits: int) -> tuple[np.array, float]:
    # Calculate the scale
    alpha = np.max(np.abs(params))
    scale = alpha / (2**(bits-1)-1)
    lower_bound = -2**(bits-1)
    upper_bound = 2**(bits-1)-1
    # Quantize the parameters
    quantized = clamp(np.round(params / scale), lower_bound, upper_bound).astype(np.int32)
    return quantized, scale

def quantization_error(params: np.array, params_q: np.array):
    # calculate the MSE
    return np.mean((params - params_q)**2)

(asymmetric_q, asymmetric_scale, asymmetric_zero) = asymmetric_quantization(params, 8)
(symmetric_q, symmetric_scale) = symmetric_quantization(params, 8)

print(f'Original:')
print(np.round(params, 2))
print('')
print(f'Asymmetric scale: {asymmetric_scale}, zero: {asymmetric_zero}')
print(asymmetric_q)
print('')
print(f'Symmetric scale: {symmetric_scale}')
print(symmetric_q)

Original:
[142.8  -49.78   0.   105.22  16.48 -18.28 -19.5  -48.78  32.92  77.53
 141.8   42.16 119.59   0.16 115.18  -8.   -23.92 107.23  23.84  40.68]

Asymmetric scale: 0.7552156862745099, zero: 66.0
[255   0  66 205  88  42  40   1 110 169 254 122 224  66 219  55  34 208
  98 120]

Symmetric scale: 1.1244094488188978
[127 -44   0  94  15 -16 -17 -43  29  69 126  37 106   0 102  -7 -21  95
  21  36]


In [ ]:
# Dequantize the parameters back to 32 bits
params_deq_asymmetric = asymmetric_dequantize(asymmetric_q, asymmetric_scale, asymmetric_zero)
params_deq_symmetric = symmetric_dequantize(symmetric_q, symmetric_scale)

print(f'Original:')
print(np.round(params, 2))
print('')
print(f'Dequantize Asymmetric:')
print(np.round(params_deq_asymmetric,2))
print('')
print(f'Dequantize Symmetric:')
print(np.round(params_deq_symmetric, 2))

Original:
[142.8  -49.78   0.   105.22  16.48 -18.28 -19.5  -48.78  32.92  77.53
 141.8   42.16 119.59   0.16 115.18  -8.   -23.92 107.23  23.84  40.68]

Dequantize Asymmetric:
[142.74 -49.84   0.   104.97  16.61 -18.13 -19.64 -49.09  33.23  77.79
 141.98  42.29 119.32   0.   115.55  -8.31 -24.17 107.24  24.17  40.78]

Dequantize Symmetric:
[142.8  -49.47   0.   105.69  16.87 -17.99 -19.11 -48.35  32.61  77.58
 141.68  41.6  119.19   0.   114.69  -7.87 -23.61 106.82  23.61  40.48]


In [ ]:
# Calculate the quantization error
print(f'{"Asymmetric error: ":>20}{np.round(quantization_error(params, params_deq_asymmetric), 2)}')
print(f'{"Symmetric error: ":>20}{np.round(quantization_error(params, params_deq_symmetric), 2)}')

  Asymmetric error: 0.05
   Symmetric error: 0.11


# PTQ

In [ ]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import os

In [ ]:
# Make torch deterministic
_ = torch.manual_seed(0)

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# Load the MNIST dataset
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# Create a dataloader for the training
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load the MNIST test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Define the device
device = "cpu"

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 531kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.6MB/s]


In [ ]:
class VerySimpleNet(nn.Module):
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(VerySimpleNet,self).__init__()
        self.linear1 = nn.Linear(28*28, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()

    def forward(self, img):
        x = img.view(-1, 28*28)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x

In [ ]:
net = VerySimpleNet().to(device)

In [ ]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()

        loss_sum = 0
        num_iterations = 0

        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            output = net(x.view(-1, 28*28))
            loss = cross_el(output, y)
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

def print_size_of_model(model):
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

MODEL_FILENAME = 'simplenet_ptq.pt'

if Path(MODEL_FILENAME).exists():
    net.load_state_dict(torch.load(MODEL_FILENAME))
    print('Loaded model from disk')
else:
    train(train_loader, net, epochs=1)
    # Save the model to disk
    torch.save(net.state_dict(), MODEL_FILENAME)

Epoch 1: 100%|██████████| 6000/6000 [00:35<00:00, 167.10it/s, loss=0.223]


In [ ]:
def test(model: nn.Module, total_iterations: int = None):
    correct = 0
    total = 0

    iterations = 0

    model.eval()

    with torch.no_grad():
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = model(x.view(-1, 784))
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct +=1
                total +=1
            iterations += 1
            if total_iterations is not None and iterations >= total_iterations:
                break
    print(f'Accuracy: {round(correct/total, 3)}')

In [ ]:
# Print the weights matrix of the model before quantization
print('Weights before quantization')
print(net.linear1.weight)
print(net.linear1.weight.dtype)

Weights before quantization
Parameter containing:
tensor([[ 7.4288e-05,  1.9500e-02, -2.9053e-02,  ...,  2.2277e-02,
          4.0743e-03,  2.4020e-03],
        [-1.5416e-02, -1.0620e-02, -6.0812e-03,  ..., -1.5903e-02,
         -1.6012e-03, -2.5585e-02],
        [ 2.0023e-02,  5.5070e-02,  6.8960e-03,  ...,  1.9828e-02,
          4.1354e-02,  4.8199e-02],
        ...,
        [ 4.9189e-02,  5.2900e-02,  1.8281e-02,  ...,  1.2918e-02,
          3.2172e-02, -4.7867e-03],
        [-9.3254e-03, -1.2189e-03,  3.0838e-02,  ...,  1.1121e-02,
          1.1175e-02,  1.0678e-02],
        [ 1.7474e-02,  1.2149e-02, -2.0101e-03,  ...,  3.4410e-02,
         -1.4872e-02,  5.2539e-03]], device='cuda:0', requires_grad=True)
torch.float32


In [ ]:
print('Size of the model before quantization')
print_size_of_model(net)

Size of the model before quantization
Size (KB): 361.465


In [ ]:
print(f'Accuracy of the model before quantization: ')
test(net)

Accuracy of the model before quantization: 


Testing: 100%|██████████| 1000/1000 [00:02<00:00, 334.01it/s]

Accuracy: 0.964


In [ ]:
#Image → flatten → QUANTIZE → Linear → ReLU → Linear → ReLU → Linear → DEQUANTIZE → output

In [ ]:
class QuantizedVerySimpleNet(nn.Module):
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(QuantizedVerySimpleNet,self).__init__()
        self.quant = torch.quantization.QuantStub()#Converts float input → integer (quantized) at the start.
        #x_q = round(x / scale) + zero_point
        self.linear1 = nn.Linear(28*28, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()#: Converts integer → float at the end.

    def forward(self, img):
        x = img.view(-1, 28*28) #flatten
        x = self.quant(x)#Float pixels → quantized int8 values.
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        x = self.dequant(x)
        return x

INPUT (float image)
    ↓
[ QuantStub ]  ← "compress to int8"
    ↓
Linear1 + ReLU   } all in quantized
Linear2 + ReLU   } integer math
Linear3            }
    ↓
[ DeQuantStub ] ← "back to float"
    ↓
OUTPUT (float scores for 10 classes)

In [ ]:
net_quantized = QuantizedVerySimpleNet().to(device)#Builds your network with QuantStub and DeQuantStub.
# Copy weights from unquantized model
net_quantized.load_state_dict(net.state_dict())#Copies all learned weights from the original trained model (net) into net_quantized.
net_quantized.eval()#Turns off training behaviors (like dropout, batch norm updates).

net_quantized.qconfig = torch.ao.quantization.default_qconfig
#use int8
#use asymmetric mapping (like your black-image formula with scale + zero_point)
net_quantized = torch.ao.quantization.prepare(net_quantized) # Insert observers
net_quantized
"""
What is an observer?
An observer is a watcher that records:

minimum value seen
maximum value seen
for each tensor (inputs, weights, activations).

From those min/max values, PyTorch later computes:

scale (s)
zero_point (z)
"""

/tmp/ipykernel_492/1743408182.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  net_quantized = torch.ao.quantization.prepare(net_quantized) # Insert observers


QuantizedVerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)

In [ ]:
test(net_quantized)

Testing: 100%|██████████| 1000/1000 [00:03<00:00, 286.12it/s]

Accuracy: 0.964


In [ ]:
print(f'Check statistics of the various layers')
net_quantized

Check statistics of the various layers


QuantizedVerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=-0.4242129623889923, max_val=2.821486711502075)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=-54.36076736450195, max_val=35.85297393798828)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=-25.926666259765625, max_val=27.277896881103516)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=-28.4958438873291, max_val=21.301502227783203)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)

In [ ]:
net_quantized = torch.ao.quantization.convert(net_quantized)

/tmp/ipykernel_492/1503456306.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  net_quantized = torch.ao.quantization.convert(net_quantized)


In [ ]:
print(f'Check statistics of the various layers')
net_quantized

Check statistics of the various layers


QuantizedVerySimpleNet(
  (quant): Quantize(scale=tensor([0.0256], device='cuda:0'), zero_point=tensor([17], device='cuda:0'), dtype=torch.quint8)
  (linear1): QuantizedLinear(in_features=784, out_features=100, scale=0.7103444337844849, zero_point=77, qscheme=torch.per_tensor_affine)
  (linear2): QuantizedLinear(in_features=100, out_features=100, scale=0.41893357038497925, zero_point=62, qscheme=torch.per_tensor_affine)
  (linear3): QuantizedLinear(in_features=100, out_features=10, scale=0.3921051025390625, zero_point=73, qscheme=torch.per_tensor_affine)
  (relu): ReLU()
  (dequant): DeQuantize()
)

In [ ]:
# Print the weights matrix of the model after quantization
print('Weights after quantization')
print(torch.int_repr(net_quantized.linear1.weight()))

Weights after quantization
tensor([[ 0,  4, -6,  ...,  5,  1,  1],
        [-3, -2, -1,  ..., -3,  0, -6],
        [ 4, 12,  1,  ...,  4,  9, 10],
        ...,
        [11, 11,  4,  ...,  3,  7, -1],
        [-2,  0,  7,  ...,  2,  2,  2],
        [ 4,  3,  0,  ...,  7, -3,  1]], device='cuda:0', dtype=torch.int8)


In [ ]:
print('Original weights: ')
print(net.linear1.weight)
print('')
print(f'Dequantized weights: ')
print(torch.dequantize(net_quantized.linear1.weight()))
print('')

Original weights: 
Parameter containing:
tensor([[ 7.4288e-05,  1.9500e-02, -2.9053e-02,  ...,  2.2277e-02,
          4.0743e-03,  2.4020e-03],
        [-1.5416e-02, -1.0620e-02, -6.0812e-03,  ..., -1.5903e-02,
         -1.6012e-03, -2.5585e-02],
        [ 2.0023e-02,  5.5070e-02,  6.8960e-03,  ...,  1.9828e-02,
          4.1354e-02,  4.8199e-02],
        ...,
        [ 4.9189e-02,  5.2900e-02,  1.8281e-02,  ...,  1.2918e-02,
          3.2172e-02, -4.7867e-03],
        [-9.3254e-03, -1.2189e-03,  3.0838e-02,  ...,  1.1121e-02,
          1.1175e-02,  1.0678e-02],
        [ 1.7474e-02,  1.2149e-02, -2.0101e-03,  ...,  3.4410e-02,
         -1.4872e-02,  5.2539e-03]], device='cuda:0', requires_grad=True)

Dequantized weights: 
tensor([[ 0.0000,  0.0185, -0.0278,  ...,  0.0231,  0.0046,  0.0046],
        [-0.0139, -0.0093, -0.0046,  ..., -0.0139,  0.0000, -0.0278],
        [ 0.0185,  0.0555,  0.0046,  ...,  0.0185,  0.0416,  0.0463],
        ...,
        [ 0.0509,  0.0509,  0.0185,  ...,  0

In [ ]:
print('Size of the model after quantization')
print_size_of_model(net_quantized)

Size of the model after quantization
Size (KB): 95.861


In [ ]:
print('Testing the model after quantization')
test(net_quantized)

Testing the model after quantization


Testing:   0%|          | 0/1000 [00:00<?, ?it/s]


RuntimeError: Unable to find an engine to execute this computation Quantized Linear Cudnn